In [98]:
import os
import pandas as pd
import joblib
from FeaturesTemporais import FeaturesTemporais
from sklearn.ensemble import RandomForestRegressor
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_squared_error, root_mean_squared_error

def get_csv_root():
    try:
        BASE_DIR = os.path.dirname(os.path.abspath(__file__))
    except NameError:
        BASE_DIR = os.getcwd()
    return os.path.join(BASE_DIR, "..", "processado")

In [99]:
df_treinamento = pd.read_csv(os.path.join(get_csv_root(), "treinamento_final.csv"))
df_testes = pd.read_csv(os.path.join(get_csv_root(), "testes_final.csv"))

display(df_treinamento.columns)

Index(['id_patient', 'ts', 'metodo_medida', 'glucose_level', 'basal',
       'bolus_type', 'bolus_dose', 'meal_type', 'meal_carbs', 'meal_time',
       'sleeping', 'sleep_quality', 'exercise_intensity', 'doing_exercise',
       'target', 'glucose_lag_5', 'glucose_lag_10', 'glucose_lag_15',
       'glucose_lag_20', 'glucose_lag_25', 'glucose_lag_30', 'glucose_delta_5',
       'glucose_delta_15', 'glucose_delta_30', 'glucose_ma_15',
       'glucose_ma_30', 'glucose_std_30'],
      dtype='str')

In [100]:
# ---- 1. Carregar ----
df_treinamento = pd.read_csv(os.path.join(get_csv_root(), "treinamento_final.csv"))
df_testes = pd.read_csv(os.path.join(get_csv_root(), "testes_final.csv"))

columns_used = ['id_patient', 'ts', 'metodo_medida', 'glucose_level', 'basal',
       'bolus_type', 'bolus_dose', 'meal_type', 'meal_carbs', 'meal_time',
       'sleeping', 'sleep_quality', 'exercise_intensity', 'doing_exercise',
       'target', 'glucose_lag_5', 'glucose_lag_10', 'glucose_lag_15',
       'glucose_lag_20', 'glucose_lag_25', 'glucose_lag_30', 'glucose_delta_5',
       'glucose_delta_15', 'glucose_delta_30', 'glucose_ma_15',
       'glucose_ma_30', 'glucose_std_30']

df_tr = df_treinamento[columns_used].copy()
df_val = df_testes[columns_used].copy()

df_tr = df_tr.rename(columns={"ts": "timestamp"})
df_val = df_val.rename(columns={"ts": "timestamp"})


In [101]:
# ---- 2. Regra de negócio (igual antes) ----
for df in (df_tr, df_val):
    df['meal_carbs'] = df['meal_carbs'].fillna(0)
    df['exercise_intensity'] = df['exercise_intensity'].fillna(0)
    df.loc[df["meal_carbs"] == 0, "meal_type"] = "Fasting"

# ---- 2.5. Remover linhas sem alvo ----
df_tr = df_tr.dropna(subset=["target"]).reset_index(drop=True)
df_val = df_val.dropna(subset=["target"]).reset_index(drop=True)

In [102]:
# ---- 3. Separar X e y ----
y_tr = df_tr["target"]
X_tr = df_tr.drop(columns=["target"])

y_val = df_val["target"]
X_val = df_val.drop(columns=["target"])

display(X_tr)

,id_patient,timestamp,metodo_medida,glucose_level,basal,bolus_type,bolus_dose,meal_type,meal_carbs,meal_time,...,glucose_lag_15,glucose_lag_20,glucose_lag_25,glucose_lag_30,glucose_delta_5,glucose_delta_15,glucose_delta_30,glucose_ma_15,glucose_ma_30,glucose_std_30
0,540,2027-05-19 11:20:00,finger_stick,80.0,0.95,NaN,0.0,Fasting,0.0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,540,2027-05-19 11:40:00,CGM,72.0,0.95,NaN,0.0,Fasting,0.0,NaN,...,NaN,NaN,NaN,NaN,-8.0,NaN,NaN,77.333333,NaN,NaN
2,540,2027-05-19 11:45:00,CGM,68.0,0.95,NaN,0.0,Fasting,0.0,NaN,...,80.0,NaN,NaN,NaN,-4.0,-12.0,NaN,73.333333,NaN,NaN
3,540,2027-05-19 11:50:00,CGM,65.0,0.95,NaN,0.0,Fasting,0.0,NaN,...,80.0,80.0,NaN,NaN,-3.0,-15.0,NaN,68.333333,NaN,NaN
4,540,2027-05-19 11:55:00,CGM,63.0,0.95,NaN,0.0,Fasting,0.0,NaN,...,72.0,80.0,80.0,NaN,-2.0,-9.0,NaN,65.333333,71.333333,7.366591
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
128316,596,2027-05-26 23:05:00,CGM,163.0,0.45,NaN,0.0,Fasting,0.0,NaN,...,158.0,157.0,155.0,152.0,2.0,5.0,11.0,161.000000,158.833333,2.857738
128317,596,2027-05-26 23:10:00,CGM,166.0,0.45,NaN,0.0,Fasting,0.0,NaN,...,159.0,158.0,157.0,155.0,3.0,7.0,11.0,163.333333,160.666667,3.386247
128318,596,2027-05-26 23:15:00,CGM,168.0,0.45,NaN,0.0,Fasting,0.0,NaN,...,161.0,159.0,158.0,157.0,2.0,7.0,11.0,165.666667,162.500000,3.937004
128319,596,2027-05-26 23:20:00,CGM,167.0,0.45,NaN,0.0,Fasting,0.0,NaN,...,163.0,161.0,159.0,158.0,-1.0,4.0,9.0,167.000000,164.000000,3.577709


In [103]:
# ---- 4. Identificar colunas numéricas e categóricas ----
num_cols = X_tr.select_dtypes(include=["int64", "float64", "bool"]).columns.tolist()
cat_cols = [c for c in X_tr.select_dtypes(include=["object", "category"]).columns if c != "timestamp"]


print("Numéricas:", num_cols)
print("Categóricas:", cat_cols)

Numéricas: ['id_patient', 'glucose_level', 'basal', 'bolus_dose', 'meal_carbs', 'meal_time', 'sleeping', 'sleep_quality', 'exercise_intensity', 'doing_exercise', 'glucose_lag_5', 'glucose_lag_10', 'glucose_lag_15', 'glucose_lag_20', 'glucose_lag_25', 'glucose_lag_30', 'glucose_delta_5', 'glucose_delta_15', 'glucose_delta_30', 'glucose_ma_15', 'glucose_ma_30', 'glucose_std_30']
Categóricas: ['metodo_medida', 'bolus_type', 'meal_type']


C:\Users\PMI\AppData\Local\Temp\ipykernel_33344\284490340.py:3: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols = [c for c in X_tr.select_dtypes(include=["object", "category"]).columns if c != "timestamp"]


In [ ]:
# ---- 5. Pipeline ----
preprocessador = ColumnTransformer(
    transformers=[
        ("num", SimpleImputer(strategy="median"), num_cols),
        ("cat", Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", OneHotEncoder(handle_unknown="ignore")),
        ]), cat_cols),
    ],
    remainder="passthrough",   # <- aqui
)

pipeline = Pipeline([
    ("tempo", FeaturesTemporais(coluna_timestamp="timestamp")),
    ("preprocessador", preprocessador),
    ("rf", RandomForestRegressor(
        n_estimators=1000,
        min_samples_leaf=10,
        random_state=0,
        n_jobs=-1,
    )),
])

pipeline.fit(X_tr, y_tr)


In [ ]:
# ---- 6. Avaliar ----
p = pipeline.predict(X_val)
mse = mean_squared_error(y_val, p)
rmse = root_mean_squared_error(y_val, p)
print(f"MSE: {mse}")
print(f"RMSE: {rmse}")

MSE: 1204.159524196982
RMSE: 34.70100177512145


In [ ]:
# 1. Contexto de escala
print(y_tr.describe())
print(f"std target: {y_tr.std():.2f}")

# 2. Baselines
media_tr = y_tr.mean()
rmse_media = root_mean_squared_error(y_val, [media_tr] * len(y_val))
rmse_persist = root_mean_squared_error(y_val, X_val["glucose_level"])
print(f"RMSE chutando a média:      {rmse_media:.2f}")
print(f"RMSE persistência:          {rmse_persist:.2f}")
print(f"RMSE do seu RF:             {rmse:.2f}")

# 3. Importância
rf = pipeline.named_steps["rf"]
nomes = pipeline.named_steps["preprocessador"].get_feature_names_out()
importancias = pd.Series(rf.feature_importances_, index=nomes).sort_values(ascending=False)
print(importancias.head(25))

count    126848.000000
mean        158.701816
std          60.432205
min          39.000000
25%         113.000000
50%         151.000000
75%         196.000000
max         400.000000
Name: target, dtype: float64
std target: 60.43
RMSE chutando a média:      60.22
RMSE persistência:          38.55
RMSE do seu RF:             34.70
num__glucose_level                7.654638e-01
num__glucose_lag_30               6.576429e-02
remainder__hora_cos               3.605958e-02
remainder__hora_sin               3.422527e-02
num__basal                        3.039720e-02
num__id_patient                   2.436545e-02
remainder__dia_sin                1.601059e-02
remainder__dia_cos                1.096208e-02
remainder__madrugada              6.593164e-03
remainder__pos_cafe               2.110780e-03
remainder__pos_almoco             1.664397e-03
remainder__fim_de_semana          1.595060e-03
remainder__pos_jantar             1.486479e-03
num__exercise_intensity           1.278123e-03
cat__meto

In [ ]:
# ---- 7. Salvar ----
joblib.dump(pipeline, "modelo_rf.joblib")
print("Modelo salvo em modelo_rf.joblib")

Modelo salvo em modelo_rf.joblib
